In [1]:
from cryosparc.tools import CryoSPARC
import os
from pydantic import BaseModel
import pandas as pd
import yaml
from datetime import datetime
from typing import Optional
import json
import bson
from cryosparc.models.session import Session

In [2]:
from cryosparc.json_util import api_object_hook
from cryosparc import registry

In [3]:
remote_cs = CryoSPARC("https://cryosparcbeta.niehs.nih.gov", email="bogomolovaa2@nih.gov")
remote_cs.test_connection()
# api_client = remote_cs.api()

Success: Connected to CryoSPARC API at https://cryosparcbeta.niehs.nih.gov


True

## Cryosparc API Structure

### `remote_cs` class has:
* method for programmatic control - `api` (https://tools.cryosparc.com/api/cryosparc.api.html#module-cryosparc.api)
* methods for navigation in data structures like:
    * `find_jobs` - search through all jobs
    * `find_projects` - search through all projects
    * `find_workspaces` - search through all workspaces
    * `list_files` - return files under project 
    * `list_assets` - return assets under project
    * `download_assets`, `download_dataset`, `download_file`, `download_mrc` - downloads corresponding element locally, those methods can be applied to both to projects and jobs. They are dealing with data structure on file system and provide convinient way to navigate and read from file system.


### CryoSPARC provides different tools for access project/jobs/sessions. 
* For control over process - use api method like:
    ``` 
    remote_cs.api.sessions.find(...)
    remote_cs.api.projects.find(...)
    remote_cs.api.jobs.find(...)
    ```
    * useful methods for **sessions** :
        - `compact_session(project, session)`
        - `find_exposure_groups(project, session)`
        - `mark session_completed(project)`
    * useful methods for **projects**:
        - `archive(project)`
        - `cleanup_data(project)`
        - `detach(project)`
        - `find()`
        - `get_directory(project)`
        - `symlink(project)` - create a symlink
    * useful methods for **jobs**:
        - `add_tag(project, job, tag)`
        - `clear_intermediate_results(project)`
        - `find_output_result(project, job)`
        - `get_directory(project, job)`

* For general information like status and stats (https://tools.cryosparc.com/api/cryosparc.controllers.html):
    ```
    remote_cs.find_project()
    remote_cs.find_project().model

    remote_cs.find_workspace()
    remote_cs.find_workspace().model

    remote_cs.find_job()
    remote_cs.find_job().model

    ```

* Comparison for `remote_cs.api` vs direct `remote_cs.find_project()`:
    - some methods may exist in both objects
    - in general 'remote_cs.api` provides more functionality over control of project/jobs/session
    - in general 'remote_cs.find_project()`/job/session provides more information about orgatinazational structure, allows list and download files as well as access statistics.



### Typical flow can look like that:
1) Working on specific project/job/session

    ```
    project10 = remote_cs.find_project("P10")
    project10.list_files('S1')
    project10.download_file('S1/exposures.bson', target="sample.bson")
    with open("sample.bson", "rb") as f:
        data_p10 = bson.loads(f.read())

    job = remote_cs.find_job("P10", "J1")
    job.list_files()
    job.download_file('job.json', target="job_sample.json")
    with open("job_sample.json", "rb") as f:
        job_config = json.loads(f.read())

    job_child = remote_cs.find_job("P10", "J30")
    all_exp = job_child.download_dataset('P10_S1_all_live_exposures.cs') 
    ```

2) Working on jobs of specific type
    ```
    jobs = remote_cs.api.jobs.find(type=['import_movies'], limit=500)
    jobs_live = remote_cs.api.jobs.find(type=['live_session'], limit=500)
    ```

## Examples:

### Get Live Session Informatioin

In [7]:
import math

def human_readable(size):
    if size == 0:
        return "0 B"

    units = ["B", "KB", "MB", "GB", "TB", "PB"]
    i = int(math.log(size, 1000))
    scaled = size / (1000 ** i)
    return f"{scaled:.2f} {units[i]}"

In [8]:
class SessionBasic(BaseModel):
    id: str
    created_at: datetime
    uid: str
    project_uid: str
    session_uid: str
    title: str
    status: str
    paused_at: Optional[datetime]
    live_session_job_uid: str
    class2D_job: Optional[str]
    class2D_num_particles_accepted: int
    class2D_num_particles_rejected: int
    class2D_num_particles_seen: int
    refine_job: Optional[str]
    refine_num_particles_in: int
    select2D_job: Optional[str]
    abinit_job: Optional[str]
    frames: int
    total_exposures: int
    total_accepted: int
    total_rejected: int
    raw: str
    metadata: str
    micrographs: str
    particles: str
    thumbnails: str

    @classmethod
    def create_stats(cls, data: Session):
        data = data.model_dump()
        data_stats = {i:human_readable(j["size"]) for i, j in data["data_management"].items() if j["status"] == "active"}
        exp_fields = [
            "file_engine_filter",
            "file_engine_watch_path_abs",
        ]
        exposure_info = {i:j for i, j in data.items() if i in exp_fields}
        exp_stats_fields = [
            "frames",
            "total_exposures",
            "total_accepted",
            "total_rejected"
        ]
        exp_stats = {i:j for i, j in data["stats"].items() if i in exp_stats_fields}
        particles_stats = {i.split('_', maxsplit=1)[-1]:j for i, j in data.items() if i.startswith("phase2")}
        finish_date = {"paused_at": data["paused_at"][-1]} if data["paused_at"] else {"paused_at": None}
        result = data | data_stats | exposure_info | exp_stats | particles_stats | finish_date
        return cls(**result)

In [ ]:
sessions = remote_cs.api.sessions.find()
df_stats = pd.DataFrame([SessionBasic.create_stats(i).model_dump() for i in sessions])

In [10]:
df_stats

,id,created_at,uid,project_uid,session_uid,title,status,paused_at,live_session_job_uid,class2D_job,...,abinit_job,frames,total_exposures,total_accepted,total_rejected,raw,metadata,micrographs,particles,thumbnails
0,65dfc0c6bab43acc35ba73d7,2024-02-28 23:24:54.408000+00:00,W2,P7,S1,MN3_2,completed,2024-03-02 03:25:09.411000+00:00,J2,J68,...,J87,60,4657,4200,457,2.03 TB,33.08 GB,3.53 TB,640.52 GB,706.01 MB
1,65e2f53abab43acc35baa480,2024-03-02 09:45:30.009000+00:00,W5,P8,S2,live,completed,2024-03-03 22:33:40.074000+00:00,J34,J64,...,J66,60,2230,2227,0,1.20 TB,17.13 GB,1.69 TB,229.49 GB,334.20 MB
2,65ecb2b9bab43acc35bab449,2024-03-09 19:04:25.120000+00:00,W3,P7,S2,MN3_2_Recollect,paused,2024-03-11 02:11:06.171000+00:00,J207,J229,...,J165,60,5590,4829,761,2.48 TB,38.83 GB,4.24 TB,509.38 GB,841.43 MB
3,66005e02bab43acc35baca54,2024-03-24 17:08:18.301000+00:00,W2,P10,S1,HoM7_2,paused,2024-04-01 14:25:39.694000+00:00,J1,J28,...,J25,60,8013,4582,3431,1.74 TB,23.63 GB,1.54 TB,4.86 TB,1.13 GB
4,662bbc97bab43acc35baeb9e,2024-04-26 14:39:19.528000+00:00,W6,P7,S3,FRO76_2,paused,2024-04-29 12:55:10.947000+00:00,J311,J329,...,J335,60,3596,3596,0,676.03 GB,9.28 GB,693.15 GB,343.51 GB,534.34 MB
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
71,69862a9fee0857ded3e6e283,2026-02-06 17:53:35.972000+00:00,W1,P39,S1,Fast2_2_81kx,compacted,2026-02-10 16:46:57.821000+00:00,J1,NaN,...,NaN,60,3483,3483,0,1.03 TB,70.85 MB,0 B,0 B,0 B
72,698b613d2a4f96e5d292abe0,2026-02-10 16:47:57.083000+00:00,W1,P40,S1,44240_3_130kx,completed,2026-02-11 18:10:48.136000+00:00,J1,J6,...,J16,60,8613,6593,2019,0 B,0 B,0 B,0 B,0 B
73,6998bc758294ceedca8f4b5b,2026-02-20 19:56:37.928000+00:00,W6,P16,S5,Ribo-G4-long-mRNA_4,completed,2026-02-23 20:58:44.053000+00:00,J68,J72,...,NaN,40,20027,17648,2379,0 B,0 B,0 B,0 B,0 B
74,699c7c0384c066879822f485,2026-02-10 16:47:57.083000+00:00,W1,P41,S1,44240_3_130kx,compacted,2026-02-11 18:10:48.136000+00:00,J1,J6,...,J16,60,8613,6593,2019,1.92 TB,694.51 MB,192.76 MB,0 B,0 B


### Find all symlinks in projects

In [ ]:
from cryosparc.models.job import Job

In [5]:
jobs = remote_cs.api.jobs.find(type=['import_movies'], limit=500)
jobs_live = remote_cs.api.jobs.find(type=['live_session'], limit=500)

In [ ]:
class MoviesImportJobs(BaseModel):
    project_uid: str
    workspace_uid: list[str]
    job_uid: str
    session_uid: Optional[str] = ''
    verify: bool = False
    process_dir: Optional[str]
    data_group: Optional[str]
    data_project: Optional[str]
    num_movies: int

    @classmethod
    def create_from_jobs(cls, data: Job):
        raw_path = data.spec.params.blob_paths
        group, raw_path = group_from_raw(raw_path)

        new_instance = {
            "project_uid": data.project_uid,
            "workspace_uid": data.workspace_uids,
            "job_uid": data.uid,
            "process_dir": remote_cs.api.jobs.get_directory(data.project_uid, data.uid),
            "data_group": group,
            "data_project": raw_path,
            "num_movies": data.spec.outputs.root['imported_movies'].num_items
        }
        return cls(**new_instance)
    
    @classmethod
    def create_from_live_jobs(cls, data: Job):
        proj_uid = data.project_uid
        sess_uid = data.spec.params.session_uid
        proj_live = remote_cs.find_project(proj_uid)
        if not proj_live.model.detached:
            live_files = proj_live.list_files(sess_uid)
            if f"{sess_uid}/import_movies" in live_files:
                process_dir = f"{proj_live.dir}/{sess_uid}/import_movies"
                verify = False
            else:
                process_dir = f"{proj_live.dir}/{sess_uid}"
                verify = True
        else:
            process_dir = f"{proj_live.dir}/{sess_uid}"
            verify = True

        sess_files = remote_cs.api.sessions.find_exposure_groups(proj_uid, sess_uid)
        path = f"{sess_files[0].file_engine_watch_path_abs}/{sess_files[0].file_engine_filter}"
        group, raw_path = group_from_raw(path)

        new_instance = {
            "project_uid": proj_uid,
            "workspace_uid": data.workspace_uids,
            "job_uid": data.uid,
            "session_uid": sess_uid,
            "process_dir": f"{process_dir}/{sess_files[0].file_engine_filter}",
            "verify": verify,
            "data_group": group,
            "data_project": raw_path,
            "num_movies": sess_files[0].num_exposures_found
        }
        return cls(**new_instance)


def group_from_raw(path):
    data_dir = '/ddn/gs1/project/cryoemCore/data/'
    if data_dir in path:
        raw_start, _, raw_path = path.partition(data_dir)
        raw_split = raw_path.split('/', maxsplit=2)
        group = raw_split[0] if raw_split[0] != 'projects_niehs' else raw_split[1]
    else:
        group = path
    return group, path


In [8]:
jobs_list = [MoviesImportJobs.create_from_jobs(j) for j in jobs if len(j.build_errors) == 0]
jobs_live_list = [MoviesImportJobs.create_from_live_jobs(j) for j in jobs_live if j.spec.params.session_uid != j.workspace_uids[0]]

In [9]:
df_jobs = pd.DataFrame([i.model_dump() for i in jobs_list])
df_jobs_live = pd.DataFrame([i.model_dump() for i in jobs_live_list])

In [10]:
df_all_jobs = pd.concat([df_jobs, df_jobs_live], axis=0)

In [11]:
df_all_jobs

,project_uid,job_uid,session_uid,verify,process_dir,data_group,data_project,num_movies
0,P1,J1,,False,/ddn/gs1/project/cryoemCore/process/CryoSPARC-...,PedersenL,/ddn/gs1/project/cryoemCore/data/projects_nieh...,2284
1,P1,J4,,False,/ddn/gs1/project/cryoemCore/process/CryoSPARC-...,PedersenL,/ddn/gs1/project/cryoemCore/data/projects_nieh...,4577
2,P1,J31,,False,/ddn/gs1/project/cryoemCore/process/CryoSPARC-...,PedersenL,/ddn/gs1/project/cryoemCore/data/projects_nieh...,366
3,P1,J34,,False,/ddn/gs1/project/cryoemCore/process/CryoSPARC-...,PedersenL,/ddn/gs1/project/cryoemCore/data/projects_nieh...,5000
4,P1,J122,,False,/ddn/gs1/project/cryoemCore/process/CryoSPARC-...,PedersenL,/ddn/gs1/project/cryoemCore/data/projects_nieh...,6980
...,...,...,...,...,...,...,...,...
66,P33,J226,S6,False,/ddn/gs1/project/cryoemCore/process/CryoSPARC-...,BorgniaM,/ddn/gs1/project/cryoemCore/data/projects_nieh...,3271
67,P33,J235,S7,False,/ddn/gs1/project/cryoemCore/process/CryoSPARC-...,BorgniaM,/ddn/gs1/project/cryoemCore/data/projects_nieh...,88
68,P36,J31,S2,False,/ddn/gs1/project/cryoemCore/process/CryoSPARC-...,BorgniaM,/ddn/gs1/project/cryoemCore/data/projects_nieh...,12342
69,P33,J247,S8,False,/ddn/gs1/project/cryoemCore/process/CryoSPARC-...,BorgniaM,/ddn/gs1/project/cryoemCore/data/projects_nieh...,3704


In [12]:
with pd.ExcelWriter('cryosparc_data_relation.xlsx', engine='openpyxl') as writer:
    df_all_jobs.to_excel(writer, index=False)